## Tutorial for using the preprocessing pipline of `ra_utils`

This is a simple tutorial which shows how to use a part of the package. 
- load some sample data (where the path to the data is defined in the `.env` file).
- apply some custom transforms (for our package)
- visualize the results (using our package)

Prerequesits: 
- **Data**

    Assuming you mounted the directory of the data. Maybe with
    
    `sshfs cn:/project/liver_RADIPOP /home/clemens/data/cirdata/liver`

    *Note:* You might consider putting the your path into your `.env` file. E.g. 
    ```bash
    DATA_ROOT_DIRECTORY=/home/clemens/data/cirdata
    DATA_PATH_LIVER_DATA=${DATA_ROOT_DIRECTORY}/liver/nnUNet_raw/Dataset125_LSS
    ```
    which we now with `config = dotenv_values()`. 
    If you prefere these variables to be global you may load them with `load_dotenv()`

In [ ]:
# Standard libraries
import os
from pathlib import Path

# Third-party libraries
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset

# MONAI libraries
from monai.apps import download_and_extract
from monai.config import print_config
from monai.data import DataLoader, CacheDataset
from monai.metrics import ROCAUCMetric
from monai.networks.nets import DenseNet121
from monai.transforms import Compose, EnsureChannelFirstd, LoadImaged, Orientationd, ScaleIntensityRanged, Spacingd, BorderPadd
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

# Additional libraries
import ra_utils
import ra_utils.features.transforms 
from ra_utils.features.transforms import SpatialCropToROId

import ra_utils.visualization
import ra_utils.visualization.semantic_segmentation

# Load .env
from dotenv import load_dotenv, dotenv_values
config = dotenv_values()


Define the paths where the sample images are located.

In [ ]:

patient_ids = ["patient0117"]  # resizing is expensive ...  let's just do this for a single CT for demonstration

# get list of files 
im_  =  [Path(config["DATA_PATH_LIVER_DATA"]) / "imagesAll" / f"{patient_id}_0000.nii.gz" for patient_id in patient_ids] 
lab_ =  [Path(config["DATA_PATH_LIVER_DATA"]) / "labelsAll" / f"{patient_id}.nii.gz" for patient_id in patient_ids] 
files = [{"image": im, "label": lab} for im, lab in zip(im_, lab_)]

# check existence
for file in files: 
    assert os.path.isfile(file["image"]), f"file {file['image']} does not exist"
    assert os.path.isfile(file["label"]), f"file {file['label']} does not exist"

### Define a composition of transforms, including our *custom transform* to crop it to the ROI

In [ ]:
# Define a transform pipeline
transforms = Compose([
        LoadImaged(keys=['image', 'label']),
        EnsureChannelFirstd(keys=['image', 'label']),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        ScaleIntensityRanged(keys=["image"], a_min=-200, a_max=200, b_min=0.0, b_max=1.0, clip=True),
        Spacingd(
            keys=["image", "label"],
            pixdim=(0.71875, 0.71875, 12),
            mode=("bilinear", "nearest"),
        ),
        BorderPadd(keys=["image", "label"], spatial_border=100), # make sure that there is something to crop 
        # crop in centeral way around ROI
        ra_utils.features.transforms.SpatialCropToROId(keys=["image", "label"], 
                          roi_size=(343, 241, 14), 
                          pedantic=False, verbose=True, use_center_instead=[False, False, False]), 
    ])


    
ds = CacheDataset(
    data=files, 
    transform = transforms,
    cache_rate=1.0,
)

loader = DataLoader(
   ds, batch_size=1, shuffle=True, num_workers=8, pin_memory=True
)


### Display the result of the transform:

In [ ]:
# plot some slice
data = ds[0]
z_index = 10
image_xy_slice = data["image"][0, :, :, z_index].detach().numpy()
label_xy_slice = data["label"][0, :, :, z_index].detach().numpy()

color_dictionary = {
    'violet': (170/255, 48/255, 127/255, 0.45),
    'blue':   (0, 0, 1, 0.45),
}

# rotate image to the usual view
image_xy_slice = image_xy_slice[::-1,::-1].transpose()
label_xy_slice = label_xy_slice[::-1,::-1].transpose()

#                                                           ct               liver                        spleen                      
ra_utils.visualization.semantic_segmentation.overlay_masks(image_xy_slice, [label_xy_slice==1,           label_xy_slice==2], 
                                                                            [color_dictionary['violet'],  color_dictionary['blue']]);

We find that the CT was nicely cropped to the ROI defined by the segmentation mask. 